# Decision model 2: two-step decision
First pick the application, then pick an action from only that application's actions.

In [ ]:
import json
import time
from typing import cast

from dotenv import load_dotenv
from IPython.display import JSON, display
from typesafe_sdk import Choice, ChoiceAnswer, TypeSafeClient

from just_ask_1 import VALID_APPLICATIONS

load_dotenv()

In [ ]:
transcript = "Hey Google, navigate to Ara."

In [ ]:
# Opened once and reused across cells; closed at the end of the notebook
client = TypeSafeClient(model="jev-latest").__enter__()

## Call 1: which application?

In [ ]:
app_questions = {
    "application": Choice(
        instructions="Which application does the user want to use?",
        criteria={name: app["description"] for name, app in VALID_APPLICATIONS.items()},
    ),
}
app_questions

In [ ]:
start = time.perf_counter()
app_response = client.system_one(state=transcript, questions=app_questions)
print(f"Application call took {time.perf_counter() - start:.2f}s")
app_response

In [ ]:
app_answer = cast(ChoiceAnswer, app_response.answers["application"])
app_name = app_answer.choice
app = VALID_APPLICATIONS[app_name]
app_answer

## Call 2: which action, within the chosen application?

In [ ]:
action_questions = {
    "action": Choice(
        instructions=(
            f"The user is using the {app_name} application "
            f"({app['description']}). Which action do they want?"
        ),
        criteria={
            action_name: action_name.replace("_", " ") for action_name in app["actions"]
        },
    ),
}
action_questions

In [ ]:
start = time.perf_counter()
action_response = client.system_one(state=transcript, questions=action_questions)
print(f"Action call took {time.perf_counter() - start:.2f}s")
action_response

In [ ]:
action_answer = cast(ChoiceAnswer, action_response.answers["action"])
action_name = action_answer.choice
action_answer

## Result

In [ ]:
print(f"target_application: {app_answer.choice} ({app_answer.confidence:.2f})")
print(f"target_action:      {action_answer.choice} ({action_answer.confidence:.2f})")

In [ ]:
client.__exit__(None, None, None)